In [23]:
import pandas as pd
import os
import glob
import re
import random
from tqdm import tqdm

# 设置随机种子以保证实验可复现
random.seed(42)

# 绝对路径配置
BASE_DIR = r'D:\Project_Github\Indo-Pacific-humpback-dolphin'
DATA_DIR = r'D:\Project_Github\Indo-Pacific-humpback-dolphin\00_Data'
CLICK_TRAINS_CSV = r'D:\Project_Github\Indo-Pacific-humpback-dolphin\00_Data\ClickTrains.csv'
POS_ROOT_DIR = r'D:\Project_Github\Indo-Pacific-humpback-dolphin\00_Data\ClickTrains'
NEG_CSV_PATH = r'D:\Project_Github\Indo-Pacific-humpback-dolphin\04_Classification\filtered_underwater_negatives.csv'
OUTPUT_DIR = r'D:\Project_Github\Indo-Pacific-humpback-dolphin\06_DataSplit'

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

print(f"Chico, 基础路径已确认。输出目录为: {OUTPUT_DIR}")

Chico, 基础路径已确认。输出目录为: D:\Project_Github\Indo-Pacific-humpback-dolphin\06_DataSplit


In [24]:
import pandas as pd
import os
import glob
from tqdm import tqdm

# 读取ClickTrains.csv
df_click = pd.read_csv(CLICK_TRAINS_CSV)

# 存储结构: {ori_file_num: [path1, path2, ...]}
pos_samples_by_ori = {}

print("正在提取正样本路径...")
# 使用 tqdm 显示总体进度条
for index, row in tqdm(df_click.iterrows(), total=df_click.shape[0], desc="Processing Positive Samples"):
    # Chico, 这里加入判断：如果 Train_num(No.) 或 Ori_file_num(No.) 为空，则跳过
    if pd.isna(row['Train_num(No.)']) or pd.isna(row['Ori_file_num(No.)']):
        continue
        
    try:
        train_num = int(row['Train_num(No.)'])
        ori_file_num = int(row['Ori_file_num(No.)'])
        
        # 构造文件夹名称，如 PulseTrain_001
        folder_name = f"PulseTrain_{train_num:03d}"
        folder_path = os.path.join(POS_ROOT_DIR, folder_name)
        
        if os.path.exists(folder_path):
            # 寻找所有 Pulse_*.wav 文件，排除 PulseTrain.wav
            pulses = glob.glob(os.path.join(folder_path, "Pulse_*.wav"))
            
            if ori_file_num not in pos_samples_by_ori:
                pos_samples_by_ori[ori_file_num] = []
            pos_samples_by_ori[ori_file_num].extend(pulses)
            
    except ValueError:
        # 以防万一存在无法转换为int的其他非数字字符，在此捕获并跳过
        continue

# 统计每个Ori_file的正样本数
pos_counts = {k: len(v) for k, v in pos_samples_by_ori.items()}
print(f"Chico, 正样本提取完成。跳过了缺失值行。共涉及 {len(pos_counts)} 个有效 Ori_file。")

正在提取正样本路径...


Processing Positive Samples: 100%|██████████| 834/834 [00:00<00:00, 3001.30it/s]

Chico, 正样本提取完成。跳过了缺失值行。共涉及 34 个有效 Ori_file。


In [25]:
# 读取负样本CSV
df_neg = pd.read_csv(NEG_CSV_PATH)

# 存储结构: {ori_file_num: [path1, path2, ...]}
neg_samples_by_ori = {}

print("正在解析并采样负样本...")
# 正则匹配提取 Ori_Recording_XX 中的数字
pattern = re.compile(r'Ori_Recording_(\d+)')

for path in df_neg['path']:
    match = pattern.search(path)
    if match:
        ori_file_num = int(match.group(1))
        if ori_file_num not in neg_samples_by_ori:
            neg_samples_by_ori[ori_file_num] = []
        neg_samples_by_ori[ori_file_num].append(path)

# 进行局部平衡采样
balanced_neg_samples_by_ori = {}
for ori_num, neg_paths in tqdm(neg_samples_by_ori.items(), desc="Balancing Negatives"):
    # 获取对应的正样本数量
    target_count = pos_counts.get(ori_num, 0)
    
    if target_count > 0:
        # 如果负样本够，则采样等量；如果不够，则全部取走
        sample_size = min(len(neg_paths), target_count)
        balanced_neg_samples_by_ori[ori_num] = random.sample(neg_paths, sample_size)
    else:
        # 如果该Ori_file没有正样本，则不在此任务中包含其负样本
        balanced_neg_samples_by_ori[ori_num] = []

print("Chico, 负样本平衡采样完成。")

正在解析并采样负样本...


Balancing Negatives: 100%|██████████| 34/34 [00:00<00:00, 6799.52it/s]

Chico, 负样本平衡采样完成。


In [26]:
# 定义划分规则
val_folds = {
    1: [10],
    2: [6],
    3: [29,30,31,32],
    4: [2],
    5: [8],
    6: [14],
    7: [21,22,23,24,25,26],
    8: [17,16,3,4,5],
    9: [28,27,20,35,15],
    10: [7,34,13,1,19]
}
test_ori_nums = [9, 11, 18]

# 获取所有出现的 Ori_file_num
all_ori_nums = set(pos_samples_by_ori.keys()) | set(balanced_neg_samples_by_ori.keys())
# 训练集候选（排除测试集）
train_val_candidates = all_ori_nums - set(test_ori_nums)

summary_stats = []

def save_to_csv(data_list, filename):
    df = pd.DataFrame(data_list, columns=['path', 'label'])
    df.to_csv(os.path.join(OUTPUT_DIR, filename), index=False)
    return len(df[df['label'] == 1]), len(df[df['label'] == 0])

# 1. 处理测试集
test_data = []
for num in test_ori_nums:
    test_data.extend([(p, 1) for p in pos_samples_by_ori.get(num, [])])
    test_data.extend([(p, 0) for p in balanced_neg_samples_by_ori.get(num, [])])
t_pos, t_neg = save_to_csv(test_data, "test_set.csv")
summary_stats.append({"Fold": "TestSet", "Train_Pos": 0, "Train_Neg": 0, "Val_Pos": t_pos, "Val_Neg": t_neg})

# 2. 处理10折交叉验证
for fold_idx, v_nums in tqdm(val_folds.items(), desc="Generating Folds"):
    val_data = []
    train_data = []
    
    # 验证集所属的 Ori_file_num
    current_val_nums = set(v_nums)
    # 训练集所属的 Ori_file_num (该折验证集以外的，且不是测试集的)
    current_train_nums = train_val_candidates - current_val_nums
    
    # 构建验证集
    for num in current_val_nums:
        val_data.extend([(p, 1) for p in pos_samples_by_ori.get(num, [])])
        val_data.extend([(p, 0) for p in balanced_neg_samples_by_ori.get(num, [])])
        
    # 构建训练集
    for num in current_train_nums:
        train_data.extend([(p, 1) for p in pos_samples_by_ori.get(num, [])])
        train_data.extend([(p, 0) for p in balanced_neg_samples_by_ori.get(num, [])])
        
    v_pos, v_neg = save_to_csv(val_data, f"fold_{fold_idx}_val.csv")
    tr_pos, tr_neg = save_to_csv(train_data, f"fold_{fold_idx}_train.csv")
    
    summary_stats.append({
        "Fold": f"Fold_{fold_idx}",
        "Train_Pos": tr_pos, "Train_Neg": tr_neg,
        "Val_Pos": v_pos, "Val_Neg": v_neg
    })

print("Chico, 所有 CSV 文件已生成。")

Generating Folds: 100%|██████████| 10/10 [00:00<00:00, 27.92it/s]

Chico, 所有 CSV 文件已生成。


In [27]:
df_stats = pd.DataFrame(summary_stats)
print("\nChico, 数据集划分统计结果如下：")
print(df_stats.to_string(index=False))

# 保存统计报告
df_stats.to_csv(os.path.join(OUTPUT_DIR, "split_statistics.csv"), index=False)


Chico, 数据集划分统计结果如下：
   Fold  Train_Pos  Train_Neg  Val_Pos  Val_Neg
TestSet          0          0     1103      731
 Fold_1      14540       8702     2775     1216
 Fold_2      15575       8178     1740     1740
 Fold_3      15419       8992     1896      926
 Fold_4      15766       8968     1549      950
 Fold_5      15335       8886     1980     1032
 Fold_6      15318       8713     1997     1205
 Fold_7      15121       9060     2194      858
 Fold_8      16219       9232     1096      686
 Fold_9      16453       9419      862      499
Fold_10      16102       9118     1213      800
